# Цепочка обработки чата: от сырых сообщений до оценки

Полный путь обработки чата менеджера с клиентом:  
сырые сообщения → группировка → фильтрация → проверки по времени → ИИ-анализ → итоговая оценка.

**Все данные выдуманные (обезличены). Запросы к ИИ — настоящие (DeepSeek API через curl).**

> Вызов ИИ через `curl` — обычный HTTP POST, аналогичный `curl_exec` в PHP.

### Ключевое: `evidence_msg_ids`
ИИ возвращает не просто цитату, а **массив ID сообщений** как доказательство.  
CRM может подсветить конкретные сообщения. ID верифицируемый — цитата может быть перефразирована, а ID нет.

---
## 0. Настройка

In [7]:
import os, json, hashlib, subprocess, time as _time
from datetime import datetime

API_KEY = os.environ.get("DEEPSEEK_API_KEY", "")
API_URL = "https://api.deepseek.com/chat/completions"
MODEL = "deepseek-chat"
MAX_RETRIES = 3

print(f"Модель:  {MODEL}")
print(f"URL:     {API_URL}")
print(f"Retries: {MAX_RETRIES}")
print(f"Ключ:    ***{API_KEY[-4:] if API_KEY else 'НЕ ЗАДАН'}")

Модель:  deepseek-chat
URL:     https://api.deepseek.com/chat/completions
Retries: 3
Ключ:    ***b681


---
## 1. Сырые сообщения из CRM

CRM присылает **отдельные сообщения** — каждое со своим `message_id`.  
Этот ID потом используется ИИ для привязки доказательств (`evidence_msg_ids`).

### Откуда мы знаем кто менеджер, а кто клиент?

Не угадываем — CRM сама присылает **направление** каждого сообщения:

```
  incoming  = сообщение пришло извне (с номера клиента)   → Клиент
  outgoing  = сообщение отправлено из CRM/Wazzup          → Менеджер
```

Это стандартное поле любого чат-API (Wazzup, Avito, Telegram, amoCRM).  
Wazzup — прослойка между менеджером и клиентом, он всегда знает направление.  
Как `from` и `to` в почте — не нужно угадывать, кто написал.

In [8]:
raw_messages = [
    {"message_id": "msg_001", "manager_id": "mgr_42", "client_external_id": "cli_wa_79211234567",
     "source": "whatsapp", "sender": "client",
     "text": "Здравствуйте, видел рекламу ЖК Северная долина, хотел узнать цены на двушки",
     "sent_at": "2026-05-25 10:02:00"},
    {"message_id": "msg_002", "manager_id": "mgr_42", "client_external_id": "cli_wa_79211234567",
     "source": "whatsapp", "sender": "manager",
     "text": "Добрый день! Да, двухкомнатные от 7,2 млн. Подскажите, рассматриваете ипотеку или за наличные?",
     "sent_at": "2026-05-25 10:03:15"},
    {"message_id": "msg_003", "manager_id": "mgr_42", "client_external_id": "cli_wa_79211234567",
     "source": "whatsapp", "sender": "client",
     "text": "Ипотеку скорее всего, семейную если возможно",
     "sent_at": "2026-05-25 10:05:30"},
    {"message_id": "msg_004", "manager_id": "mgr_42", "client_external_id": "cli_wa_79211234567",
     "source": "whatsapp", "sender": "manager",
     "text": "Отлично, семейная от 6%. Оставьте ваш номер телефона — я подберу подходящие варианты и перезвоню с деталями",
     "sent_at": "2026-05-25 10:06:10"},
    {"message_id": "msg_005", "manager_id": "mgr_42", "client_external_id": "cli_wa_79211234567",
     "source": "whatsapp", "sender": "client",
     "text": "+7 921 123-45-67",
     "sent_at": "2026-05-25 10:07:00"},
    {"message_id": "msg_006", "manager_id": "mgr_42", "client_external_id": "cli_wa_79211234567",
     "source": "whatsapp", "sender": "manager",
     "text": "Записал! Перезвоню в течение часа. Меня зовут Алексей, менеджер отдела продаж. Если что — пишите, на связи",
     "sent_at": "2026-05-25 10:08:20"}
]

# Индекс ID -> сообщение (для валидации позже)
MSG_INDEX = {m["message_id"]: m for m in raw_messages}

print(f"Получено сообщений: {len(raw_messages)}")
print(f"Все ID: {list(MSG_INDEX.keys())}\n")
for m in raw_messages:
    who = "Менеджер" if m["sender"] == "manager" else "Клиент  "
    t = m["sent_at"].split(" ")[1][:5]
    print(f"  {m['message_id']}  {t}  {who}  {m['text'][:70]}")

Получено сообщений: 6
Все ID: ['msg_001', 'msg_002', 'msg_003', 'msg_004', 'msg_005', 'msg_006']

  msg_001  10:02  Клиент    Здравствуйте, видел рекламу ЖК Северная долина, хотел узнать цены на д
  msg_002  10:03  Менеджер  Добрый день! Да, двухкомнатные от 7,2 млн. Подскажите, рассматриваете 
  msg_003  10:05  Клиент    Ипотеку скорее всего, семейную если возможно
  msg_004  10:06  Менеджер  Отлично, семейная от 6%. Оставьте ваш номер телефона — я подберу подхо
  msg_005  10:07  Клиент    +7 921 123-45-67
  msg_006  10:08  Менеджер  Записал! Перезвоню в течение часа. Меня зовут Алексей, менеджер отдела


---
## 2. Группировка в диалоги

Ключ диалога = `менеджер + клиент + мессенджер` (SHA-256, первые 16 символов).  
30 сообщений от одного клиента одному менеджеру в WhatsApp = **один диалог**.

In [9]:
def make_conversation_key(manager_id, client_id, source):
    raw = f"{manager_id}|{client_id}|{source}"
    return hashlib.sha256(raw.encode()).hexdigest()[:16]

def group_into_dialogs(messages):
    dialogs = {}
    for msg in messages:
        key = make_conversation_key(msg["manager_id"], msg["client_external_id"], msg["source"])
        if key not in dialogs:
            dialogs[key] = {
                "conversation_key": key,
                "manager_id": msg["manager_id"],
                "client_id": msg["client_external_id"],
                "source": msg["source"],
                "messages": []
            }
        dialogs[key]["messages"].append(msg)
    for d in dialogs.values():
        d["messages"].sort(key=lambda m: m["sent_at"])
    return dialogs

dialogs = group_into_dialogs(raw_messages)
dialog = list(dialogs.values())[0]

print(f"{len(raw_messages)} сообщений -> {len(dialogs)} диалог")
print(f"  Ключ:      {dialog['conversation_key']}")
print(f"  Менеджер:  {dialog['manager_id']}")
print(f"  Клиент:    {dialog['client_id']}")
print(f"  Источник:  {dialog['source']}")
print(f"  Сообщений: {len(dialog['messages'])}")

6 сообщений -> 1 диалог
  Ключ:      fef6de9f39a790ca
  Менеджер:  mgr_42
  Клиент:    cli_wa_79211234567
  Источник:  whatsapp
  Сообщений: 6


---
## 3. Текст диалога для ИИ (с message_id!)

Каждая строка начинается с `[message_id]` — ИИ будет ссылаться на эти ID в ответе.

In [10]:
def build_dialog_text(dialog):
    lines = []
    for m in dialog["messages"]:
        t = m["sent_at"].split(" ")[1][:5]
        who = "Менеджер" if m["sender"] == "manager" else "Клиент"
        lines.append(f"[{m['message_id']}] [{t}] {who}: {m['text']}")
    return "\n".join(lines)

dialog_text = build_dialog_text(dialog)

print("Текст диалога (с ID сообщений):")
print("=" * 70)
print(dialog_text)
print("=" * 70)
print(f"Длина: {len(dialog_text)} символов")

Текст диалога (с ID сообщений):
[msg_001] [10:02] Клиент: Здравствуйте, видел рекламу ЖК Северная долина, хотел узнать цены на двушки
[msg_002] [10:03] Менеджер: Добрый день! Да, двухкомнатные от 7,2 млн. Подскажите, рассматриваете ипотеку или за наличные?
[msg_003] [10:05] Клиент: Ипотеку скорее всего, семейную если возможно
[msg_004] [10:06] Менеджер: Отлично, семейная от 6%. Оставьте ваш номер телефона — я подберу подходящие варианты и перезвоню с деталями
[msg_005] [10:07] Клиент: +7 921 123-45-67
[msg_006] [10:08] Менеджер: Записал! Перезвоню в течение часа. Меня зовут Алексей, менеджер отдела продаж. Если что — пишите, на связи
Длина: 609 символов


---
## 4. Фильтрация

In [11]:
MIN_MESSAGES = 4

n = len(dialog["messages"])
senders = set(m["sender"] for m in dialog["messages"])
ok = n >= MIN_MESSAGES and len(senders) >= 2

print(f"Сообщений: {n} (мин {MIN_MESSAGES}), участников: {len(senders)} (мин 2)")
print(f"Результат: {'ПРОХОДИТ' if ok else 'ОТКЛОНЁН'}")

Сообщений: 6 (мин 4), участников: 2 (мин 2)
Результат: ПРОХОДИТ


---
## 5a. Проверки по времени (код, без ИИ)

Часть правил — чистая арифметика по таймстампам. ИИ не нужен.

### Правило: «Менеджер ответил в течение 2 минут»

Берём первое сообщение клиента и ближайший ответ менеджера. Считаем разницу.

```
  ВРЕМЯ       КТО          ЧТО ПРОИЗОШЛО
  ─────       ───          ─────────────
  10:02:00    Клиент       Написал вопрос                     ◄── засекли время
                │
                │  прошло 75 секунд
                │
  10:03:15    Менеджер     Ответил                            ◄── засекли время
                │
                ▼
           75 сек < 120 сек = ВЫПОЛНЕНО
           Доказательство: сообщения msg_001 и msg_002
```

### Правило: «Если клиент замолчал на 5 минут — менеджер напомнил о себе»

Смотрим: после сообщения менеджера клиент молчит. Если молчит 5+ минут — проверяем, написал ли менеджер повторно.

```
  ВРЕМЯ       КТО          ЧТО ПРОИЗОШЛО
  ─────       ───          ─────────────
  10:03:15    Менеджер     Ответил клиенту
                │
                │  клиент молчит...
                │  прошло 2 мин 15 сек
                │
  10:05:30    Клиент       Написал сам            ◄── молчал меньше 5 мин,
                                                      правило не применяется


  А если бы клиент молчал дольше:

  10:03:15    Менеджер     Ответил клиенту
                │
                │  клиент молчит...
                │  прошло 5 минут — порог!
                │
                ├── Менеджер написал напоминание?
                │     ДА  = ВЫПОЛНЕНО   (доказательство: оба сообщения менеджера)
                │     НЕТ = НЕ ВЫПОЛНЕНО (доказательство: последнее сообщение менеджера)
```

In [12]:
def check_response_time(dialog, max_seconds=120):
    msgs = dialog["messages"]
    for i, msg in enumerate(msgs):
        if msg["sender"] == "client":
            for j in range(i + 1, len(msgs)):
                if msgs[j]["sender"] == "manager":
                    t1 = datetime.strptime(msg["sent_at"], "%Y-%m-%d %H:%M:%S")
                    t2 = datetime.strptime(msgs[j]["sent_at"], "%Y-%m-%d %H:%M:%S")
                    delta = (t2 - t1).total_seconds()
                    return {
                        "criterion": f"Ответ менеджера <= {max_seconds} сек",
                        "passed": delta <= max_seconds,
                        "value": f"{int(delta)} сек",
                        "evidence_msg_ids": [msg["message_id"], msgs[j]["message_id"]],
                        "detail": f"{msg['sent_at'].split(' ')[1]} -> {msgs[j]['sent_at'].split(' ')[1]}"
                    }
    return {"criterion": f"Ответ <= {max_seconds} сек", "passed": None, "value": "Н/Д", "evidence_msg_ids": []}

time_checks = [check_response_time(dialog, 120)]

print("Проверки по времени (без ИИ):")
print("=" * 60)
for c in time_checks:
    icon = "[+]" if c["passed"] else ("[-]" if c["passed"] is False else "[~]")
    print(f"  {icon} {c['criterion']} = {c['value']}")
    print(f"      evidence_msg_ids: {c['evidence_msg_ids']}")
    if "detail" in c:
        print(f"      {c['detail']}")

Проверки по времени (без ИИ):
  [+] Ответ менеджера <= 120 сек = 75 сек
      evidence_msg_ids: ['msg_001', 'msg_002']
      10:02:00 -> 10:03:15


---
## 5b. Конфигурация: чеклист, теги, данные

In [13]:
CHECKLIST = [
    {"id": 1, "name": "Запрос номера телефона",
     "rule": "Менеджер запросил телефон в первых 3 сообщениях. Если клиент сам дал — тоже ок."},
    {"id": 2, "name": "Обоснование преимуществ",
     "rule": "Менеджер привёл конкретное преимущество объекта (цена, локация, ипотека)."},
    {"id": 3, "name": "Представился / визитка",
     "rule": "Менеджер представился и/или предложил свои контактные данные."},
    {"id": 4, "name": "Грамотная речь",
     "rule": "Нет грубых орфографических ошибок. Мелкие опечатки допустимы."}
]

TAGS = [
    {"name": "ипотека", "desc": "Обсуждается ипотека"},
    {"name": "жалоба", "desc": "Клиент жалуется"},
    {"name": "повторное обращение", "desc": "Клиент обращался ранее"},
    {"name": "грубость", "desc": "Менеджер грубит"}
]

DATA_FIELDS = [
    {"field": "client_name", "desc": "Имя клиента"},
    {"field": "property", "desc": "ЖК или объект"},
    {"field": "budget", "desc": "Бюджет"},
    {"field": "mortgage_type", "desc": "Тип ипотеки"},
    {"field": "apartment_type", "desc": "Тип квартиры"}
]

print(f"Чеклист: {len(CHECKLIST)} | Теги: {len(TAGS)} | Данные: {len(DATA_FIELDS)}")

Чеклист: 4 | Теги: 4 | Данные: 5


---
## 6. Сборка промпта

В формате ответа требуем `evidence_msg_ids` — массив ID сообщений, на которые опирается вердикт.

In [14]:
checklist_text = "\n".join(f"{c['id']}. {c['name']}: {c['rule']}" for c in CHECKLIST)
tags_text = "\n".join(f"- {t['name']}: {t['desc']}" for t in TAGS)
data_text = "\n".join(f"- {d['field']}: {d['desc']}" for d in DATA_FIELDS)

valid_ids = [m["message_id"] for m in raw_messages]

prompt = f"""Ты — система оценки качества работы менеджера в текстовом чате с клиентом.

ДИАЛОГ (каждая строка начинается с [message_id]):
{dialog_text}

Допустимые message_id: {json.dumps(valid_ids)}

---

БЛОК 1 — ЧЕКЛИСТ
По каждому критерию: выполнен (1) или не выполнен (0).
В evidence_msg_ids укажи ID сообщений, которые подтверждают вердикт (одно или несколько).
В quote — короткая цитата из этих сообщений.

{checklist_text}

---

БЛОК 2 — ТЕГИ
По каждому тегу: присутствует (1) или нет (0).
Если тег присутствует — в evidence_msg_ids укажи все сообщения, где он проявляется.
Если тег отсутствует — evidence_msg_ids = [].

{tags_text}

---

БЛОК 3 — ДАННЫЕ ДИАЛОГА
Извлеки данные. Если не упоминается — НЕ включай поле.
Для каждого поля укажи evidence_msg_ids — из каких сообщений извлечены данные.

{data_text}

---

ФОРМАТ ОТВЕТА — строго JSON:
{{
  "checklist": [
    {{"id": 1, "name": "...", "passed": 1, "evidence_msg_ids": ["msg_XXX"], "quote": "цитата"}}
  ],
  "tags": [
    {{"name": "...", "value": 1, "evidence_msg_ids": ["msg_XXX", "msg_YYY"]}}
  ],
  "dialog_data": {{
    "field": {{"value": "...", "evidence_msg_ids": ["msg_XXX"]}}
  }}
}}

ВАЖНО:
- evidence_msg_ids содержит ТОЛЬКО ID из списка допустимых
- Массив может содержать 1 или несколько ID
- Только JSON, без пояснений."""

print(f"Длина промпта: {len(prompt)} символов")
print(f"\nПервые 400 символов:")
print(prompt[:400])
print("...")

Длина промпта: 2518 символов

Первые 400 символов:
Ты — система оценки качества работы менеджера в текстовом чате с клиентом.

ДИАЛОГ (каждая строка начинается с [message_id]):
[msg_001] [10:02] Клиент: Здравствуйте, видел рекламу ЖК Северная долина, хотел узнать цены на двушки
[msg_002] [10:03] Менеджер: Добрый день! Да, двухкомнатные от 7,2 млн. Подскажите, рассматриваете ипотеку или за наличные?
[msg_003] [10:05] Клиент: Ипотеку скорее всего, с
...


---
## 7. Отправка в ИИ через curl (с retry)

HTTP POST → DeepSeek. При ошибке сети, таймауте или битом JSON — автоматический retry до 3 раз.

```
POST https://api.deepseek.com/chat/completions
Authorization: Bearer sk-...
Content-Type: application/json
```

In [15]:
def call_llm(prompt, api_key, api_url, model, max_retries=3, timeout=60):
    """Отправка в ИИ с retry при ошибках сети, таймауте или битом ответе."""
    
    request_body = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 2000
    }
    
    with open("/tmp/deepseek_request.json", "w") as f:
        json.dump(request_body, f, ensure_ascii=False)
    
    for attempt in range(1, max_retries + 1):
        print(f"  Попытка {attempt}/{max_retries}...", end=" ")
        
        try:
            t0 = _time.time()
            result = subprocess.run(
                ["curl", "-s", "-X", "POST", api_url,
                 "-H", f"Authorization: Bearer {api_key}",
                 "-H", "Content-Type: application/json",
                 "-d", "@/tmp/deepseek_request.json"],
                capture_output=True, text=True, timeout=timeout
            )
            elapsed = _time.time() - t0
        except subprocess.TimeoutExpired:
            print(f"ТАЙМАУТ ({timeout} сек)")
            if attempt < max_retries:
                _time.sleep(2 * attempt)
            continue
        
        # Проверяем что curl вернул непустой ответ
        if not result.stdout.strip():
            print(f"ПУСТОЙ ОТВЕТ (curl stderr: {result.stderr[:100]})")
            if attempt < max_retries:
                _time.sleep(2 * attempt)
            continue
        
        # Парсим JSON ответа API
        try:
            api_response = json.loads(result.stdout)
        except json.JSONDecodeError as e:
            print(f"БИТЫЙ JSON ОТ API: {e}")
            if attempt < max_retries:
                _time.sleep(2 * attempt)
            continue
        
        # Проверяем что API не вернул ошибку
        if "error" in api_response:
            err = api_response["error"]
            print(f"ОШИБКА API: {err.get('message', err)}")
            if attempt < max_retries:
                _time.sleep(2 * attempt)
            continue
        
        # Извлекаем ответ модели
        try:
            raw_answer = api_response["choices"][0]["message"]["content"]
            usage = api_response.get("usage", {})
        except (KeyError, IndexError) as e:
            print(f"НЕОЖИДАННАЯ СТРУКТУРА ОТВЕТА: {e}")
            if attempt < max_retries:
                _time.sleep(2 * attempt)
            continue
        
        print(f"OK за {elapsed:.1f} сек ({usage.get('prompt_tokens', '?')}+{usage.get('completion_tokens', '?')} токенов)")
        return {"raw": raw_answer, "usage": usage, "elapsed": elapsed, "attempt": attempt}
    
    # Все попытки провалились
    return None


# Показываем curl-команду
print("Эквивалентная curl-команда:")
print(f"""curl -X POST {API_URL} \\
  -H "Authorization: Bearer $DEEPSEEK_API_KEY" \\
  -H "Content-Type: application/json" \\
  -d @/tmp/deepseek_request.json""")
print()

# Вызываем
print("Отправляем запрос:")
llm_result = call_llm(prompt, API_KEY, API_URL, MODEL, MAX_RETRIES)

if llm_result:
    raw_answer = llm_result["raw"]
    print(f"\nСырой ответ модели:")
    print("=" * 60)
    print(raw_answer)
    print("=" * 60)
else:
    print("\nВСЕ ПОПЫТКИ ПРОВАЛИЛИСЬ")
    raw_answer = None

Эквивалентная curl-команда:
curl -X POST https://api.deepseek.com/chat/completions \
  -H "Authorization: Bearer $DEEPSEEK_API_KEY" \
  -H "Content-Type: application/json" \
  -d @/tmp/deepseek_request.json

Отправляем запрос:
  Попытка 1/3... OK за 4.9 сек (918+583 токенов)

Сырой ответ модели:
```json
{
  "checklist": [
    {
      "id": 1,
      "name": "Запрос номера телефона",
      "passed": 1,
      "evidence_msg_ids": ["msg_004"],
      "quote": "Оставьте ваш номер телефона — я подберу подходящие варианты и перезвоню с деталями"
    },
    {
      "id": 2,
      "name": "Обоснование преимуществ",
      "passed": 1,
      "evidence_msg_ids": ["msg_002", "msg_004"],
      "quote": "двухкомнатные от 7,2 млн. Подскажите, рассматриваете ипотеку или за наличные? ... семейная от 6%"
    },
    {
      "id": 3,
      "name": "Представился / визитка",
      "passed": 1,
      "evidence_msg_ids": ["msg_006"],
      "quote": "Меня зовут Алексей, менеджер отдела продаж"
    },
    {
      

### Тот же запрос на PHP

```php
$ch = curl_init('https://api.deepseek.com/chat/completions');
curl_setopt_array($ch, [
    CURLOPT_POST           => true,
    CURLOPT_RETURNTRANSFER => true,
    CURLOPT_TIMEOUT        => 60,
    CURLOPT_HTTPHEADER     => [
        'Authorization: Bearer ' . $apiKey,
        'Content-Type: application/json',
    ],
    CURLOPT_POSTFIELDS => json_encode([
        'model'       => 'deepseek-chat',
        'messages'    => [['role' => 'user', 'content' => $prompt]],
        'temperature' => 0.1,
        'max_tokens'  => 2000,
    ]),
]);

$response = curl_exec($ch);
$httpCode = curl_getinfo($ch, CURLINFO_HTTP_CODE);

if ($httpCode !== 200 || curl_errno($ch)) {
    // retry...
}

$data   = json_decode($response, true);
$answer = $data['choices'][0]['message']['content'];
$result = json_decode($answer, true); // <-- тут чеклист, теги, данные с evidence_msg_ids
```

---
## 8. Парсинг и валидация (с retry при ошибках)

Проверяем:
1. JSON парсится?
2. Все критерии оценены?
3. `evidence_msg_ids` содержат только валидные ID?
4. Нет пустых `evidence_msg_ids` у выполненных критериев?

In [16]:
def parse_ai_response(raw):
    """Парсим JSON, убираем markdown-обёртку если есть."""
    if raw is None:
        return None
    text = raw.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0]
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"  JSON parse error: {e}")
        return None


def validate_result(result, expected_ids, valid_msg_ids):
    """Валидируем структуру, ID критериев и evidence_msg_ids."""
    errors = []
    warnings = []
    valid_set = set(valid_msg_ids)
    
    # Чеклист
    if "checklist" not in result:
        errors.append("Нет блока 'checklist'")
    else:
        got_ids = {item["id"] for item in result["checklist"]}
        missing = set(expected_ids) - got_ids
        if missing:
            errors.append(f"Не оценены критерии: {missing}")
        for item in result["checklist"]:
            if item.get("passed") not in (0, 1):
                errors.append(f"Критерий {item.get('id')}: passed должен быть 0 или 1")
            eids = item.get("evidence_msg_ids", [])
            bad = [e for e in eids if e not in valid_set]
            if bad:
                errors.append(f"Критерий {item.get('id')}: несуществующие ID: {bad}")
            if item.get("passed") == 1 and not eids:
                warnings.append(f"Критерий {item.get('id')}: выполнен, но нет evidence_msg_ids")
    
    # Теги
    if "tags" not in result:
        errors.append("Нет блока 'tags'")
    else:
        for tag in result["tags"]:
            eids = tag.get("evidence_msg_ids", [])
            bad = [e for e in eids if e not in valid_set]
            if bad:
                errors.append(f"Тег '{tag.get('name')}': несуществующие ID: {bad}")
    
    # Данные
    if "dialog_data" not in result:
        errors.append("Нет блока 'dialog_data'")
    else:
        for field, val in result["dialog_data"].items():
            if isinstance(val, dict):
                eids = val.get("evidence_msg_ids", [])
                bad = [e for e in eids if e not in valid_set]
                if bad:
                    errors.append(f"Данные '{field}': несуществующие ID: {bad}")
    
    return errors, warnings


# Парсим
result = parse_ai_response(raw_answer)

if result:
    expected = [c["id"] for c in CHECKLIST]
    valid_ids = list(MSG_INDEX.keys())
    errors, warnings = validate_result(result, expected, valid_ids)
    
    if errors:
        print("ОШИБКИ (в проде = retry):")
        for e in errors:
            print(f"  [!] {e}")
    else:
        print("Валидация пройдена")
    
    if warnings:
        print("Предупреждения:")
        for w in warnings:
            print(f"  [~] {w}")
    
    print(f"\n  Чеклист: {len(result.get('checklist', []))} критериев")
    print(f"  Теги: {len(result.get('tags', []))}")
    print(f"  Данные: {len(result.get('dialog_data', {}))} полей")
else:
    print("Парсинг не удался")

Валидация пройдена

  Чеклист: 4 критериев
  Теги: 4
  Данные: 3 полей


---
## 9. Итоговая оценка

Код (время) + ИИ (смысл) → единый результат. Везде `evidence_msg_ids`.

In [17]:
print("=" * 65)
print("              ИТОГОВАЯ ОЦЕНКА ДИАЛОГА")
print("=" * 65)
print(f"Ключ: {dialog['conversation_key']} | {dialog['source']}\n")

all_checks = []

print("--- ЧЕКЛИСТ ---")
for tc in time_checks:
    if tc["passed"] is not None:
        icon = "[+]" if tc["passed"] else "[-]"
        all_checks.append(tc["passed"])
    else:
        icon = "[~]"
    print(f"  {icon} {tc['criterion']} = {tc['value']}  (код)")
    print(f"      evidence: {tc['evidence_msg_ids']}")

if result:
    for item in result["checklist"]:
        icon = "[+]" if item["passed"] == 1 else "[-]"
        all_checks.append(item["passed"] == 1)
        print(f"  {icon} {item['name']}  (ИИ)")
        print(f"      evidence: {item.get('evidence_msg_ids', [])}")
        print(f"      quote: \"{item.get('quote', '')}\"")

passed = sum(1 for c in all_checks if c)
total = len(all_checks)
pct = round(passed / total * 100) if total else 0
print(f"\n  ИТОГО: {passed}/{total} = {pct}%\n")

print("--- ТЕГИ ---")
if result:
    for t in result["tags"]:
        if t.get("value") == 1:
            print(f"  + {t['name']}  evidence: {t.get('evidence_msg_ids', [])}")
        else:
            print(f"  - {t['name']}")

print("\n--- ДАННЫЕ ДИАЛОГА ---")
if result and result.get("dialog_data"):
    for field, val in result["dialog_data"].items():
        if isinstance(val, dict):
            print(f"  {field}: {val.get('value', val)}  evidence: {val.get('evidence_msg_ids', [])}")
        else:
            print(f"  {field}: {val}")

print("\n" + "=" * 65)

              ИТОГОВАЯ ОЦЕНКА ДИАЛОГА
Ключ: fef6de9f39a790ca | whatsapp

--- ЧЕКЛИСТ ---
  [+] Ответ менеджера <= 120 сек = 75 сек  (код)
      evidence: ['msg_001', 'msg_002']
  [+] Запрос номера телефона  (ИИ)
      evidence: ['msg_004']
      quote: "Оставьте ваш номер телефона — я подберу подходящие варианты и перезвоню с деталями"
  [+] Обоснование преимуществ  (ИИ)
      evidence: ['msg_002', 'msg_004']
      quote: "двухкомнатные от 7,2 млн. Подскажите, рассматриваете ипотеку или за наличные? ... семейная от 6%"
  [+] Представился / визитка  (ИИ)
      evidence: ['msg_006']
      quote: "Меня зовут Алексей, менеджер отдела продаж"
  [+] Грамотная речь  (ИИ)
      evidence: ['msg_002', 'msg_004', 'msg_006']
      quote: "Добрый день! Да, двухкомнатные от 7,2 млн. Подскажите, рассматриваете ипотеку или за наличные?"

  ИТОГО: 5/5 = 100%

--- ТЕГИ ---
  + ипотека  evidence: ['msg_002', 'msg_003', 'msg_004']
  - жалоба
  - повторное обращение
  - грубость

--- ДАННЫЕ ДИАЛОГА ---
  p

---
## 10. Callback в CRM

JSON для CRM. Везде `evidence_msg_ids`.  
Если есть невыполненные критерии — прикладывается полный текст диалога.

In [18]:
checklist_out = []
has_failures = False

for tc in time_checks:
    if tc["passed"] is not None:
        checklist_out.append({
            "criterion": tc["criterion"],
            "passed": tc["passed"],
            "detail": tc["value"],
            "evidence_msg_ids": tc["evidence_msg_ids"]
        })
        if not tc["passed"]: has_failures = True

if result:
    for item in result["checklist"]:
        checklist_out.append({
            "criterion": item["name"],
            "passed": bool(item["passed"]),
            "quote": item.get("quote", ""),
            "evidence_msg_ids": item.get("evidence_msg_ids", [])
        })
        if not item["passed"]: has_failures = True

tags_out = []
if result:
    for t in result["tags"]:
        if t.get("value") == 1:
            tags_out.append({"name": t["name"], "evidence_msg_ids": t.get("evidence_msg_ids", [])})

data_out = {}
if result:
    for field, val in result.get("dialog_data", {}).items():
        if isinstance(val, dict):
            data_out[field] = val
        else:
            data_out[field] = {"value": val, "evidence_msg_ids": []}

callback = {
    "conversation_key": dialog["conversation_key"],
    "manager_id": dialog["manager_id"],
    "score_percent": pct,
    "passed": passed,
    "total": total,
    "checklist": checklist_out,
    "tags": tags_out,
    "dialog_data": data_out,
}

if has_failures:
    callback["full_dialog_text"] = dialog_text

print("JSON для CRM (callback):")
print(json.dumps(callback, ensure_ascii=False, indent=2))

JSON для CRM (callback):
{
  "conversation_key": "fef6de9f39a790ca",
  "manager_id": "mgr_42",
  "score_percent": 100,
  "passed": 5,
  "total": 5,
  "checklist": [
    {
      "criterion": "Ответ менеджера <= 120 сек",
      "passed": true,
      "detail": "75 сек",
      "evidence_msg_ids": [
        "msg_001",
        "msg_002"
      ]
    },
    {
      "criterion": "Запрос номера телефона",
      "passed": true,
      "quote": "Оставьте ваш номер телефона — я подберу подходящие варианты и перезвоню с деталями",
      "evidence_msg_ids": [
        "msg_004"
      ]
    },
    {
      "criterion": "Обоснование преимуществ",
      "passed": true,
      "quote": "двухкомнатные от 7,2 млн. Подскажите, рассматриваете ипотеку или за наличные? ... семейная от 6%",
      "evidence_msg_ids": [
        "msg_002",
        "msg_004"
      ]
    },
    {
      "criterion": "Представился / визитка",
      "passed": true,
      "quote": "Меня зовут Алексей, менеджер отдела продаж",
      "evidenc

---
## Резюме

| Шаг | Что | Кто | evidence_msg_ids |
|-----|-----|-----|------------------|
| 1 | CRM прислал 6 сообщений | CRM | — |
| 2 | Группировка в 1 диалог | Код | — |
| 3 | Текст с [message_id] | Код | ID встроены в текст |
| 4 | Фильтрация | Код | — |
| 5a | Время ответа 75 сек | Код | `[msg_001, msg_002]` |
| 5b | Промпт с ID | Код | ID в промпте |
| 6 | curl → DeepSeek (retry) | HTTP | — |
| 7 | Парсинг + валидация ID | Код | проверяем что ID существуют |
| 8 | Итог: 5/5 = 100% | Код | на каждом результате |
| 9 | Callback JSON | HTTP | CRM подсветит сообщения |